# Sentinel-2 NDVI -> COG -> GeoLibre

**The honest architecture here:** GeoLibre is the *display and exploration*
layer, not the raster engine. The Python API has no `add_netcdf` / `add_zarr`
(I checked the installed package — those are Add Data menu features only).

So the pipeline is the one you already run for ALPINE-WATCH:

    STAC search -> odc-stac -> xarray -> NDVI -> COG -> GeoLibre

You keep xarray for the heavy lifting and gain a real GIS front end that
publishes to a URL. Two ways to compute the index; do both and compare.


In [ ]:
import numpy as np
import xarray as xr
import rioxarray  # noqa: F401 -- registers the .rio accessor
import pystac_client
import planetary_computer
from odc.stac import stac_load
from geolibre import Map

## 1. AOI and search

Swap `BBOX` for whatever you're actually after. This one is a chunk of the
Sierra Nevada foothills; a good NDVI target because the oak woodland / grassland
mosaic has a huge seasonal swing.


In [ ]:
BBOX = [-121.2, 38.6, -120.4, 39.2]   # [w, s, e, n]
DATE = "2025-05-01/2025-06-30"

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX,
    datetime=DATE,
    query={"eo:cloud_cover": {"lt": 10}},
)

items = list(search.items())
print(f"{len(items)} scenes")
for it in items[:5]:
    print(f"  {it.id}  cloud={it.properties['eo:cloud_cover']:.1f}%")

## 2. Load red + NIR

B04 = red, B08 = NIR. `chunks={}` keeps it lazy through dask so you're not
pulling the whole stack into memory.


In [ ]:
ds = stac_load(
    items,
    bands=["B04", "B08"],
    bbox=BBOX,
    resolution=20,          # metres; 10 is native for these bands, 20 is faster
    crs="EPSG:3857",        # Web Mercator so GeoLibre serves it without reprojection
    chunks={},
    groupby="solar_day",
)

ds

## 3. NDVI

`(NIR - Red) / (NIR + Red)`. Cast to float first — the raw bands are uint16 and
integer division will silently ruin your day.


In [ ]:
red = ds["B04"].astype("float32")
nir = ds["B08"].astype("float32")

ndvi = (nir - red) / (nir + red)
ndvi = ndvi.where(np.isfinite(ndvi))
ndvi.name = "NDVI"

# Median across the time dimension -- a cheap, effective cloud/shadow filter
ndvi_median = ndvi.median(dim="time", skipna=True).compute()
ndvi_median.rio.write_crs("EPSG:3857", inplace=True)
ndvi_median.rio.write_nodata(np.nan, inplace=True)

print(ndvi_median.shape)
ndvi_median.plot(cmap="RdYlGn", vmin=-0.2, vmax=0.9, figsize=(9, 7))

## 4. Write a COG

GeoLibre reads COGs over HTTP range requests, so a properly tiled + overviewed
file streams instead of downloading.


In [ ]:
from pathlib import Path

Path("../data").mkdir(exist_ok=True)
out = "../data/ndvi_foothills_2025.tif"

ndvi_median.rio.to_raster(
    out,
    driver="COG",
    compress="DEFLATE",
    blocksize=512,
    overview_resampling="average",
    dtype="float32",
)

print(out)

## 5. Display it

`add_raster` / `add_cog` accept a **local** path on the kernel host — the
bundled localhost server hands it to the app. Two caveats worth internalising:

1. It only works where the **browser can reach the kernel's localhost** (local
   Jupyter, VS Code). On Colab/JupyterHub/remote, pass a hosted URL instead.
2. The served URL is **session-scoped**. A project saved with a local raster
   will *not* restore it when reopened. For anything durable — i.e. anything
   you publish — pass a hosted URL.


In [ ]:
m = Map(center=(-120.8, 38.9), zoom=9, basemap="dark",
        height="750px", layout="full", theme="dark")
m

In [ ]:
ndvi_layer = m.add_raster(
    out,
    name="NDVI (median, May-Jun 2025)",
    colormap="rdylgn",
    rescale=[[-0.2, 0.9]],
)

m.add_colorbar(
    colormap="rdylgn",
    vmin=-0.2, vmax=0.9,
    label="NDVI",
    position="bottom-right",
)

## 6. Swipe comparison

The obvious next move: NDVI against true colour, or spring against late summer.
`split_map` takes the layer ids returned by the `add_*` calls.


In [ ]:
# Build a second date range, write a second COG, then:
# early = m.add_raster("../data/ndvi_foothills_spring.tif", name="Spring")
# late  = m.add_raster("../data/ndvi_foothills_summer.tif", name="Late summer")
# m.split_map(early, late, orientation="vertical", position=50)

## 7. Now do it the other way

Load the raw B04/B08 COGs into the app and run **Processing → Raster →
Spectral Index**, which has NDVI/NDWI/EVI with band presets on a rasterio
sidecar. Compare the result to what you computed above.

Worth doing once properly: if the in-app toolbox handles the scaling and
nodata the way you'd want, a lot of this notebook collapses into three clicks,
and more importantly it becomes scriptable via `run_algorithm()` — which is
what you'd actually wire into a GitHub Actions cadence.


In [ ]:
for a in m.list_algorithms():
    if any(k in str(a).lower() for k in ("ndvi", "spectral", "index")):
        print(a)

## 8. Publish

```python
m.save_project("../projects/ndvi-foothills.geolibre.json")
m.to_html("../projects/ndvi-foothills.html", title="Foothills NDVI")
```

Remember point 2 above — swap the local raster path for a hosted COG URL before
saving anything you intend to reopen or share.
